In [1]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260221+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260220+cu128. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0228 22:20:35.809000 36520 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Warning - regex did not match, patch may have failed


In [2]:
MODEL_ID      = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR    = "./orvion-planner-v1"
TRAIN_FILE    = r"D:\orvion\finetuning\datasets\planner_train.jsonl"
VAL_FILE      = r"D:\orvion\finetuning\datasets\planner_val.jsonl"
MAX_SEQ_LEN   = 1024   # plenty for NL input + numbered output
LORA_RANK     = 16     # small — task is simple, avoid overfit
LORA_ALPHA    = 32
BATCH_SIZE    = 4      # text only, no images — can push higher
GRAD_ACCUM    = 2      # effective batch = 16
EPOCHS        = 5      # small dataset, more epochs needed
LR            = 3e-4   # slightly higher than executor — simpler task
WARMUP_RATIO  = 0.1

In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype        = None,  # auto
    attn_implementation="sdpa", # Standard SDPA for stability
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260221+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [10]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]
    
def safe_format(record, idx):
    try:
        return {
            "text": tokenizer.apply_chat_template(
                record["messages"],
                tokenize=False,
                add_generation_prompt=False
            )
        }
    except Exception as e:
        print(f"\n❌ Error at record index {idx}")
        print(record)
        raise e

raw_train = load_jsonl(TRAIN_FILE)

formatted = []
for i, r in enumerate(raw_train):
    formatted.append(safe_format(r, i))

train_data = Dataset.from_list(formatted)

In [11]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

def format_record(record):
    """Apply chat template to messages."""
    return {"text": tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False
    )}

train_data = Dataset.from_list([format_record(r) for r in load_jsonl(TRAIN_FILE)])
val_data   = Dataset.from_list([format_record(r) for r in load_jsonl(VAL_FILE)])

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("Sample formatted record:")
print(train_data[0]["text"][:300])

Train: 431 | Val: 118
Sample formatted record:
<|im_start|>system
You are a UAT test planner. Convert the given test case into a numbered list of clear, atomic action steps. Each step must describe exactly one action or one verification. Output only the numbered list — no explanation, no markdown, no extra text.<|im_end|>
<|im_start|>user
1. Ver


In [12]:
trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset = train_data,
    eval_dataset  = val_data,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    packing            = True,   # pack multiple short records → more efficient
    args = TrainingArguments(
        output_dir              = OUTPUT_DIR,
        num_train_epochs        = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = 4,
        learning_rate           = LR,
        warmup_ratio            = WARMUP_RATIO,
        lr_scheduler_type       = "cosine",
        fp16                    = False,
        bf16=True,  # use bf16 if supported, otherwise fallback to fp16
        logging_steps           = 10,
        eval_strategy="epoch",
        save_strategy           = "epoch",
        load_best_model_at_end  = True,
        metric_for_best_model   = "eval_loss",
        report_to               = "none",
        seed                    = 42,
    ),
)


Unsloth: Tokenizing ["text"]: 100%|██████████| 118/118 [00:00<00:00, 4035.78 examples/s]


In [13]:
import os

# Disable Unsloth’s custom compiler kernels and xFormers entirely
os.environ["UNSLOTH_FORCE_DISABLE_COMPILER"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

# Optional: ensure PyTorch uses SDPA attention
os.environ["PYTORCH_ENABLE_SDPA"] = "1"
os.environ["PYTORCH_USE_FLASH_ATTENTION"] = "0"
os.environ["DISABLE_TRITON"] = "1"


In [14]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 431 | Num Epochs = 5 | Total steps = 135
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Epoch,Training Loss,Validation Loss
1,0.855200,0.676912
2,0.484300,0.582281
3,0.310200,0.581220
4,0.192700,0.661591
5,0.137100,0.736750


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x23a328af820>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: e8d24fef-f2d7-4091-bc64-c0282df9e89c)') - silently ignoring the lookup for the file config.json in unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit.
  warnings.warn(
c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\peft\utils\save_and_load.py:295:

TrainOutput(global_step=135, training_loss=0.49838251184534144, metrics={'train_runtime': 182.8559, 'train_samples_per_second': 11.785, 'train_steps_per_second': 0.738, 'total_flos': 3131592972008448.0, 'train_loss': 0.49838251184534144})

In [15]:
model.save_pretrained_merged(OUTPUT_DIR + "/merged", tokenizer, save_method="merged_16bit")
print(f"Saved merged model to {OUTPUT_DIR}/merged")

Found HuggingFace hub cache directory: C:\Users\prasasnna\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [07:48<00:00, 468.89s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:05<00:00,  5.84s/it]


Unsloth: Merge process complete. Saved to `d:\orvion\finetuning\orvion-planner-v1\merged`
Saved merged model to ./orvion-planner-v1/merged


In [16]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [17]:

stress_test_cases = [
    # 1. THE "LOGIC LABYRINTH" (Deep Nesting & Context Switching)
    "Go to Settings, change theme to Dark, then go to User Management, search for 'Bob', click Edit, change his role to 'Admin', then go back to Settings and verify the theme is still Dark.",
    
    # 2. THE "NEGATIVE INSTRUCTION" (Testing Hallucination)
    "Navigate to the Login page. Do NOT enter a username. Enter 'password123' in the password field. Click login. Verify that NO dashboard loads and only a 'Username required' error appears.",
    
    # 3. THE "DATA HEAVYWEIGHT" (Bulk Actions & Sorting)
    "Open the Inventory table, click 'Select All', uncheck the first three items, click 'Bulk Export', select 'XML' format, wait for the download, then click 'Clear Selection' and verify 0 items are selected.",
    
    # 4. THE "VANISHING STATE" (Persistence & Refreshing)
    "Open the New Post editor, type 'Testing persistence' in the title, click 'Save Draft', refresh the browser, verify the title is still there, then click 'Publish' and verify it appears in the 'Published' tab.",
    
    # 5. THE "BOUNDARY STRETCH" (Input Limits)
    "Navigate to the 'Bio' field. Paste exactly 1000 characters of text. Verify the character counter turns red. Try to click 'Save' and verify the button is disabled. Delete 501 characters and verify 'Save' is now enabled.",
    
    # 6. THE "MULTI-ENTITY CREATION" (Instruction Density)
    "Create a folder named 'Finance'. Inside 'Finance', create a file named 'Tax_2026'. Mark 'Tax_2026' as 'High Priority'. Add a comment 'Ready for review'. Then move the 'Finance' folder to 'Archive'.",
    
    # 7. THE "UI ANOMALY" (Responsive & Non-Standard Navigation)
    "Resize the window to 400px width. Open the hamburger menu. Click 'Profile'. Close the menu. Re-open the menu. Verify the 'Profile' link is now highlighted in blue.",
    
    # 8. THE "SILENT FAIL" (Error Handling)
    "Try to upload a file named 'virus.exe'. Verify the upload starts but fails at 50%. Verify a 'Malicious file detected' warning appears and the 'Submit' button disappears.",
    
    # 9. THE "CIRCULAR FLOW" (Back-and-Forth Navigation)
    "Go to Page 1 of the wizard. Enter 'Step1'. Go to Page 2. Enter 'Step2'. Go back to Page 1. Change 'Step1' to 'Revised'. Go forward to Page 2 and verify 'Step2' is still there.",
    
    # 10. THE "AMBIGUOUS VERB" (Semantic Complexity)
    "Provision a new cloud instance named 'Genesis-Node', toggle the 'Auto-scale' switch, set the 'Max Nodes' to 5, and then decommission the instance immediately to verify the 'Pending Deletion' status."
]

print("\n=== INFERENCE TEST ===")
for tc in stress_test_cases:
    messages = [
        {"role": "system", "content": "You are a UAT test planner. Convert the given test case into a numbered list of clear, atomic action steps. Each step must describe exactly one action or one verification. Output only the numbered list — no explanation, no markdown, no extra text."},
        {"role": "user", "content": tc},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    import torch
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False, temperature=1.0)
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(f"\nInput: {tc[:60]}...")
    print(f"Output:\n{response}")
    print("-"*40)


=== INFERENCE TEST ===

Input: Go to Settings, change theme to Dark, then go to User Manage...
Output:
1. Navigate to the Settings page
2. Click the 'Dark Mode' toggle switch
3. Navigate to the User Management list
4. Search for user account 'Bob'
5. Click the edit button next to Bob's record
6. Change the role dropdown to 'Admin'
7. Click save
8. Verify the current theme settings page loads
9. Verify the UI elements are displayed in dark mode
----------------------------------------

Input: Navigate to the Login page. Do NOT enter a username. Enter '...
Output:
1. Navigate directly to the Login page without logging in first
2. Verify the Login form is visible with username/email fields
3. Verify there are no authenticated user sessions active
4. Enter 'password123' into the password field
5. Click the Login button
6. Verify an error message 'Username or password is incorrect' is displayed
7. Verify the current page URL still contains the login path
-----------------------------------

In [18]:
model.push_to_hub(
    "sanaX3065/orivion-planner-1.5B",
    tokenizer=tokenizer,
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   0%|          | 45.7kB / 73.9MB, 12.0kB/s  





Processing Files (0 / 1)                :   2%|▏         | 1.17MB / 73.9MB,  233kB/s  


Processing Files (0 / 1)                :   2%|▏         | 1.73MB / 73.9MB,  309kB/s  
Processing Files (0 / 1)                :   4%|▍         | 2.85MB / 73.9MB,  491kB/s  

Processing Files (0 / 1)                :   5%|▍         | 3.41MB / 73.9MB,  550kB/s  
Processing Files (0 / 1)                :   6%|▌         | 4.53MB / 73.9MB,  708kB/s  
Processing Files (0 / 1)                :   8%|▊         | 5.65MB / 73.9MB,  857kB/s  
Processing Files (0 / 1)                :  11%|█▏        | 8.46MB / 73.9MB, 1.24MB/s  


Processing Files (0 / 1)                :  12%|█▏        | 9.02MB / 73.9MB, 1.22MB/s  

Processing Files (0 / 1)                :  13%|█▎        | 9.65MB / 73.9MB, 1.24MB/s  

Processing Files (0 / 1)          

Saved model to https://huggingface.co/sanaX3065/orivion-planner-1.5B
